# Qwen2.5-3B RCA Two-Task Fine-Tuning
### Two-task approach: GENERATE RCA + GENERATE CAPA
### Includes full Drive checkpointing for free Colab sessions

---
**IMPORTANT — Read before running:**
- This notebook assumes your dataset is already split into `train_task.jsonl`, `val_task.jsonl`, `test_task.jsonl` in the two-task format
- Files must be pushed to your GitHub repo at `data/processed/`
- On Session 1: run ALL cells top to bottom
- On Session 2+ (after Colab dies): run ALL cells — the resume cell will auto-detect the latest checkpoint
---

## STEP 1 — Mount Google Drive First (Always)
This must be the very first thing you do in every session.
Checkpoints save here. Without this, nothing persists.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Configure your Drive checkpoint folder here ──
CHECKPOINT_DIR = "/content/drive/MyDrive/rca_twotask_checkpoints"
ADAPTER_BACKUP_DIR = "/content/drive/MyDrive/rca_twotask_adapter_backup"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ADAPTER_BACKUP_DIR, exist_ok=True)

print("Drive mounted.")
print(f"Checkpoint dir : {CHECKPOINT_DIR}")
print(f"Adapter backup : {ADAPTER_BACKUP_DIR}")

# Show how much Drive space is free
import shutil
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive free     : {free // (1024**3)} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
Checkpoint dir : /content/drive/MyDrive/rca_twotask_checkpoints
Adapter backup : /content/drive/MyDrive/rca_twotask_adapter_backup
Drive free     : 16 GB


## STEP 2 — Detect Existing Checkpoint
Run this every session. It tells you whether to start fresh or resume.

In [2]:
import os

def get_latest_checkpoint(checkpoint_dir):
    """Returns path to latest checkpoint, or None if none exist."""
    if not os.path.exists(checkpoint_dir):
        return None
    checkpoints = [
        os.path.join(checkpoint_dir, d)
        for d in os.listdir(checkpoint_dir)
        if d.startswith("checkpoint-")
    ]
    if not checkpoints:
        return None
    return max(checkpoints, key=os.path.getmtime)

latest_checkpoint = get_latest_checkpoint(CHECKPOINT_DIR)

if latest_checkpoint:
    print(f"✅ Checkpoint found: {latest_checkpoint}")
    print("   → trainer.train() will RESUME from this checkpoint")
else:
    print("ℹ️  No checkpoint found.")
    print("   → trainer.train() will START from scratch")

✅ Checkpoint found: /content/drive/MyDrive/rca_twotask_checkpoints/checkpoint-350
   → trainer.train() will RESUME from this checkpoint


## STEP 3 — Install Unsloth

In [ ]:
# Step 1 — wipe everything conflicting
!pip uninstall -y unsloth unsloth-zoo transformers trl torchao -q

# Step 2 — install in strict order, pinned versions known to work together
!pip install -q "transformers==4.51.3"
!pip install -q "trl==1.3.0"
!pip install -q "torchao==0.9.0"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Step 3 — verify before restarting
import subprocess
result = subprocess.run(["pip", "show", "transformers", "trl", "unsloth"], capture_output=True, text=True)
print(result.stdout)

# Step 4 — restart runtime to apply clean installs
import os
os.kill(os.getpid(), 9)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## STEP 4 — Import Libraries

In [3]:
import os
import json
import torch
import random
import numpy as np
import shutil

from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, TrainerCallback

import transformers, trl, unsloth, platform

print("="*60)
print("Python       :", platform.python_version())
print("Torch        :", torch.__version__)
print("Transformers :", transformers.__version__)
print("TRL          :", trl.__version__)
print("Unsloth      :", unsloth.__version__)
print("="*60)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Python       : 3.12.13
Torch        : 2.11.0+cu128
Transformers : 5.5.0
TRL          : 0.24.0
Unsloth      : 2026.6.9


## STEP 5 — Verify GPU

In [4]:
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name       :", torch.cuda.get_device_name(0))
    print(
        "GPU Memory     :",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

!nvidia-smi

CUDA Available : True
GPU Name       : Tesla T4
GPU Memory     : 14.56 GB
Sat Jun 27 07:49:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             29W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                       

## STEP 6 — Clone Repo and Load Dataset
Your repo must have the two-task split files at `data/processed/`:
- `train_task.jsonl`
- `val_task.jsonl`
- `test_task.jsonl`

Each sample must be in this format:
```json
{"messages": [{"role": "user", "content": "TASK: GENERATE RCA\n\n..."}, {"role": "assistant", "content": "{...}"}]}
```

In [5]:
%cd /content
!rm -rf rca-slm
!git clone https://github.com/KritiP25/rca-slm.git
%cd rca-slm

/content
Cloning into 'rca-slm'...
remote: Enumerating objects: 23085, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 23085 (delta 33), reused 57 (delta 23), pack-reused 23017 (from 1)
Receiving objects: 100% (23085/23085), 132.41 MiB | 11.88 MiB/s, done.
Resolving deltas: 100% (1958/1958), done.
/content/rca-slm


In [6]:
dataset = load_dataset(
    "json",
    data_files={
        "train": "data/processed/(2)train_task.jsonl",
        "validation": "data/processed/(2)val_task.jsonl",
        "test": "data/processed/(2)test_task.jsonl",
    }
)

print(dataset)

# Sanity check: count task types in training set
rca_count = sum(
    1 for ex in dataset["train"]
    if "GENERATE RCA" in ex["messages"][0]["content"]
)
capa_count = sum(
    1 for ex in dataset["train"]
    if "GENERATE CAPA" in ex["messages"][0]["content"]
)
print(f"\nTrain — Task A (GENERATE RCA)  : {rca_count}")
print(f"Train — Task B (GENERATE CAPA) : {capa_count}")
print(f"Train — Total                  : {len(dataset['train'])}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2328
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 290
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 292
    })
})

Train — Task A (GENERATE RCA)  : 1164
Train — Task B (GENERATE CAPA) : 1164
Train — Total                  : 2328


## STEP 7 — Load Base Model
**max_seq_length is reduced to 2048** vs your original 3072.
Reason: Task A and Task B outputs are each shorter than a full report.
This saves GPU memory and speeds up training significantly on free Colab.

In [7]:
# ── Tunable parameters ──────────────────────────────────────
MAX_SEQ_LENGTH = 3072   # Reduced from 3072 — safe for two-task outputs
# ────────────────────────────────────────────────────────────

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print("Model loaded.")

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded.


## STEP 8 — Configure LoRA
**r is reduced to 8** vs your original 16.
Reason: With two clear, well-defined tasks and a structured output format,
r=8 is sufficient. It trains ~30% faster and uses less memory.

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,                              # Reduced from 16 — sufficient for two structured tasks
    lora_alpha=16,                    # Keep alpha=16 (alpha/r=2 ratio is standard)
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("LoRA configured.")
model.print_trainable_parameters()

Unsloth 2026.6.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


LoRA configured.
trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827


## STEP 9 — Apply Chat Template

In [9]:
def formatting(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = dataset.map(formatting)

print("Chat template applied.")

Map:   0%|          | 0/2328 [00:00<?, ? examples/s]

Map:   0%|          | 0/290 [00:00<?, ? examples/s]

Map:   0%|          | 0/292 [00:00<?, ? examples/s]

Chat template applied.


## STEP 10 — Inspect One Sample of Each Task Type

In [10]:
# Find first Task A and Task B example
task_a_sample = None
task_b_sample = None

for ex in dataset["train"]:
    if task_a_sample is None and "GENERATE RCA" in ex["text"]:
        task_a_sample = ex["text"]
    if task_b_sample is None and "GENERATE CAPA" in ex["text"]:
        task_b_sample = ex["text"]
    if task_a_sample and task_b_sample:
        break

print("=" * 60)
print("TASK A SAMPLE (first 1500 chars):")
print(task_a_sample[:1500] if task_a_sample else "NOT FOUND")

print("\n" + "=" * 60)
print("TASK B SAMPLE (first 1500 chars):")
print(task_b_sample[:1500] if task_b_sample else "NOT FOUND")

TASK A SAMPLE (first 1500 chars):
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
TASK: GENERATE RCA

Problem Description:
During a rolling update deployment of the Identity Session Cache Hub version 5.0.1, a catastrophic container orchestration failure caused a severe cascade of system terminations across the production cluster. The release required the deployment of twenty-four large-scale Redis-backed data processing containers designed to maintain active session states. However, the accompanying deployment manifest contained an un-optimized and overly restrictive memory resource limit allocation parameter (512Mi) that failed to account for real-world production session cache utilization trends. When the Kubernetes control plane initiated the container lifecycle changes, the newly provisioned pods instantly exceeded their memory boundaries upon loading active in-memory session indices from the database master. The Li

## STEP 11 — Adapter Backup Callback
This saves the LoRA adapter to Drive every 100 steps as a safety net.
Even if checkpoint resuming fails, you never lose more than 100 steps of work.

In [11]:
class DriveAdapterBackupCallback(TrainerCallback):
    """
    Saves a lightweight LoRA adapter backup to Drive every N steps.
    This is separate from checkpoints — it's a fallback if resume fails.
    Adapter is ~50MB, so it won't eat your Drive space.
    """
    def __init__(self, backup_dir, save_every_n_steps=100):
        self.backup_dir = backup_dir
        self.save_every_n_steps = save_every_n_steps

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.save_every_n_steps == 0 and state.global_step > 0:
            model.save_pretrained(self.backup_dir)
            tokenizer.save_pretrained(self.backup_dir)
            print(f"\n💾 Adapter backup saved at step {state.global_step} → {self.backup_dir}")

print("Callback defined.")

Callback defined.


## STEP 12 — Configure Training

Key changes from your original notebook:

| Parameter | Original | New | Why |
|---|---|---|---|
| `output_dir` | `./qwen_rca_output` | Drive path | Survives session death |
| `save_strategy` | `"no"` | `"steps"` | Enables checkpointing |
| `save_steps` | — | `50` | Save every 50 steps |
| `save_total_limit` | — | `3` | Only keep last 3 checkpoints |
| `num_train_epochs` | `3` | `2` | Dataset doubled; 2 epochs sufficient |
| `max_length` | `3072` | `2048` | Tasks are shorter; saves memory |
| `warmup_steps` | `25` | `30` | Slightly larger dataset |
| `load_best_model_at_end` | — | `True` | Auto-loads best checkpoint |


In [12]:
# STEP 12 — SFTConfig (version-corrected)

training_args = SFTConfig(

    # ── Output → MUST point to Drive ────────────────────────
    output_dir=CHECKPOINT_DIR,

    # ── Training ────────────────────────────────────────────
    num_train_epochs=2,
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    # ── Optimizer ───────────────────────────────────────────
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=30,

    # ── Mixed Precision ─────────────────────────────────────
    fp16=True,
    bf16=False,

    # ── Logging ─────────────────────────────────────────────
    logging_steps=10,

    # ── Evaluation ──────────────────────────────────────────
    eval_strategy="no",
    eval_steps=300,               # Every 200 steps, not 100

    # ── Checkpointing ───────────────────────────────────────
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    # load_best_model_at_end REMOVED — causes OOM on Colab at end of training

    # ── Dataset ─────────────────────────────────────────────
    max_length=MAX_SEQ_LENGTH,    # max_length is correct in TRL 1.7.0
    dataset_text_field="text",    # ADDED — tells trainer which column to use
    packing=False,
    dataset_num_proc=1,

    # ── Misc ────────────────────────────────────────────────
    report_to="none",
    seed=42,
)

## STEP 13 — Create Trainer

In [13]:
# STEP 13 — SFTTrainer (version-corrected)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,          # NOT processing_class — Unsloth patches SFTTrainer to expect this
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    callbacks=[
        DriveAdapterBackupCallback(
            backup_dir=ADAPTER_BACKUP_DIR,
            save_every_n_steps=100,
        )
    ],
)

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/2328 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/290 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## STEP 14 — Start or Resume Training

This cell is identical for Session 1 and all subsequent sessions.
- If a checkpoint was found in STEP 2, it resumes automatically.
- If no checkpoint, it starts fresh.

**Do not change this cell between sessions.**

In [14]:
# Re-detect checkpoint right before training (in case session was interrupted
# between STEP 2 and now)
latest_checkpoint = get_latest_checkpoint(CHECKPOINT_DIR)

if latest_checkpoint:
    print(f"Resuming from checkpoint: {latest_checkpoint}")
else:
    print("Starting training from scratch.")

trainer.train(resume_from_checkpoint=latest_checkpoint)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Resuming from checkpoint: /content/drive/MyDrive/rca_twotask_checkpoints/checkpoint-350


	eval_steps: 300 (from args) != 200 (from trainer_state.json)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,328 | Num Epochs = 2 | Total steps = 582
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 14,966,784 of 3,100,905,472 (0.48% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
360,1.210738
370,1.208661
380,1.206758
390,1.233416
400,1.177231
410,1.229916
420,1.194289
430,1.179025
440,1.168829
450,1.170857


...


💾 Adapter backup saved at step 400 → /content/drive/MyDrive/rca_twotask_adapter_backup


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/rca_twotask_checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/rca_twotask_checkpoints/checkpoint-450/tokenizer_config.json.



💾 Adapter backup saved at step 500 → /content/drive/MyDrive/rca_twotask_adapter_backup


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/rca_twotask_checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/rca_twotask_checkpoints/checkpoint-550/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/rca_twotask_checkpoints/checkpoint-582/tokenizer_config.json.


TrainOutput(global_step=582, training_loss=0.47354421869586016, metrics={'train_runtime': 7778.5503, 'train_samples_per_second': 0.599, 'train_steps_per_second': 0.075, 'total_flos': 1.3705842414890189e+17, 'train_loss': 0.47354421869586016, 'epoch': 2.0})

## STEP 15 — Save Final Adapter

Run this only after training is 100% complete.
This saves the final merged adapter — this is what you deploy.

In [15]:
FINAL_ADAPTER_DIR = "/content/drive/MyDrive/rca_twotask_final_adapter"

model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print(f"✅ Final adapter saved to: {FINAL_ADAPTER_DIR}")

# Show adapter size
total_size = sum(
    os.path.getsize(os.path.join(dirpath, f))
    for dirpath, _, files in os.walk(FINAL_ADAPTER_DIR)
    for f in files
)
print(f"   Adapter size: {total_size / 1024**2:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/rca_twotask_final_adapter/tokenizer_config.json.


✅ Final adapter saved to: /content/drive/MyDrive/rca_twotask_final_adapter
   Adapter size: 68.1 MB


## STEP 16 — Quick Inference Test

Test both tasks to confirm the model learned the task distinction.

In [16]:
FastLanguageModel.for_inference(model)

# ── Test Task A: Generate RCA ────────────────────────────────
test_input_rca = [
    {
        "role": "user",
        "content": (
            "TASK: GENERATE RCA\n\n"
            "Problem Description:\n"
            "Production database server became unresponsive at 2 AM causing "
            "complete application downtime for 3 hours.\n\n"
            "Business Impact:\n"
            "10,000 users unable to access the platform. $500K revenue loss.\n\n"
            "Technical Investigation:\n"
            "Disk space on /var/lib/mysql reached 100%. MySQL crashed due to "
            "inability to write transaction logs. No disk space alerts were configured."
        )
    }
]

inputs_rca = tokenizer.apply_chat_template(
    test_input_rca,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

print("=" * 60)
print("TASK A OUTPUT (Generate RCA):")
print("=" * 60)

outputs_rca = model.generate(
    input_ids=inputs_rca,
    max_new_tokens=800,
    temperature=0.1,
    do_sample=True,
)

print(tokenizer.decode(outputs_rca[0][inputs_rca.shape[1]:], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=800) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TASK A OUTPUT (Generate RCA):


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


{
  "issue_summary": "Production database server became unresponsive at 2 AM causing complete application downtime for 3 hours.",
  "five_why_analysis": [
    {
      "question": "1. Why was the system down?",
      "answer": "Disk space was full."
    },
    {
      "question": "2. Why was there no alert?",
      "answer": "Alerts were not configured for /var/lib/mysql"
    },
    {
      "question": "3. Why wasn't it configured?",
      "answer": "We forgot to add it to Nagios"
    }
  ],
  "root_cause_summary": {
    "statement": "The root cause was a lack of monitoring for critical filesystem partitions.",
    "root_cause_category": "Process"
  }
}


In [17]:
# ── Test Task B: Generate CAPA ───────────────────────────────
# Paste the RCA output from Task A above into approved_rca below
approved_rca = """
{
  "issue_summary": "Production database crashed at 2 AM due to disk space exhaustion on /var/lib/mysql",
  "five_why_analysis": [
    {"question": "Why did the application go down?", "answer": "MySQL crashed"},
    {"question": "Why did MySQL crash?", "answer": "Disk was full, could not write transaction logs"},
    {"question": "Why was disk full?", "answer": "Log rotation was not configured"},
    {"question": "Why was log rotation missing?", "answer": "No disk monitoring or alerting existed"},
    {"question": "Why was monitoring absent?", "answer": "Infrastructure baseline checklist did not include disk thresholds"}
  ],
  "root_cause_summary": {
    "statement": "Absence of disk space monitoring and log rotation policy caused undetected disk exhaustion.",
    "root_cause_category": "Technology"
  }
}
"""

test_input_capa = [
    {
        "role": "user",
        "content": (
            "TASK: GENERATE CAPA\n\n"
            "Problem Description:\n"
            "Production database server became unresponsive at 2 AM causing "
            "complete application downtime for 3 hours.\n\n"
            "Business Impact:\n"
            "10,000 users unable to access the platform. $500K revenue loss.\n\n"
            "Technical Investigation:\n"
            "Disk space on /var/lib/mysql reached 100%. MySQL crashed due to "
            "inability to write transaction logs. No disk space alerts were configured.\n\n"
            f"Approved Root Cause:\n{approved_rca}"
        )
    }
]

inputs_capa = tokenizer.apply_chat_template(
    test_input_capa,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

print("=" * 60)
print("TASK B OUTPUT (Generate CAPA):")
print("=" * 60)

outputs_capa = model.generate(
    input_ids=inputs_capa,
    max_new_tokens=600,
    temperature=0.1,
    do_sample=True,
)

print(tokenizer.decode(outputs_capa[0][inputs_capa.shape[1]:], skip_special_tokens=True))

Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TASK B OUTPUT (Generate CAPA):
{
  "corrective_preventive_actions": [
    {
      "action_type": "CA",
      "action_description": "Performed emergency log rotation and cleared old files to free up space",
      "action_owner": "Database Administrator"
    },
    {
      "action_type": "PA",
      "action_description": "Deployed Nagios disk space monitoring and automated log rotation scripts",
      "action_owner": "Infrastructure Team"
    }
  ],
  "lessons_learned": []
}


## STEP 17 — Optional: Download Adapter as ZIP
Only run if you want a local copy. If adapter is in Drive, you don't need this.

In [ ]:
# Uncomment to download
# import shutil
# from google.colab import files

# shutil.make_archive("rca_twotask_adapter", "zip", FINAL_ADAPTER_DIR)
# files.download("rca_twotask_adapter.zip")

---
## Checkpointing Quick Reference

### If Colab dies mid-training:
1. Open a new Colab session
2. Run ALL cells from top to bottom — nothing else needed
3. STEP 2 will auto-detect the latest checkpoint
4. STEP 14 will resume from exactly where you left off

### What's saved where:
| Location | What | When |
|---|---|---|
| `rca_twotask_checkpoints/checkpoint-N/` | Full optimizer + model state | Every 50 steps |
| `rca_twotask_adapter_backup/` | LoRA weights only (~50MB) | Every 100 steps |
| `rca_twotask_final_adapter/` | Final trained adapter | After training completes |

### Drive space estimate (14GB available):
- Each checkpoint: ~800MB–1GB
- 3 checkpoints max: ~3GB
- Adapter backup: ~50MB
- Final adapter: ~50MB
- **Total used: ~3.1GB — well within 14GB**

### Estimated training time (free T4):
- ~2300 train samples, 2 epochs, batch size 1, grad accum 8
- ~575 optimizer steps per epoch → ~1150 total steps
- ~3–4 hours total
- Free Colab T4 session limit: ~4 hours
- **Worst case: 2 sessions needed. Checkpointing handles this.**
---

In [18]:
# Run this in a new cell to check your training data
import json

empty_lessons = 0
with open("/content/rca-slm/data/raw/clean_dataset.jsonl") as f:
    for line in f:
        s = json.loads(line)
        lessons = s.get("5.0_Lessons_Learned", [])
        if not lessons or lessons == [] or lessons == [""]:
            empty_lessons += 1

print(f"Samples with empty lessons learned: {empty_lessons}/1455")

Samples with empty lessons learned: 31/1455


In [20]:
FastLanguageModel.for_inference(model)

# ── REAL TEST: Task A ─────────────────────────────────────────
test_input_rca = [
    {
        "role": "user",
        "content": (
            "TASK: GENERATE RCA\n\n"
            "Problem Description:\n"
            "On June 15, 2026, the enterprise payment processing gateway experienced "
            "a complete service outage lasting 4 hours and 23 minutes beginning at 14:32 UTC. "
            "The Stripe payment integration API began returning HTTP 503 errors across all "
            "merchant accounts. Transaction processing halted entirely for 847 active merchant "
            "accounts. The root trigger was traced to an expired SSL certificate on the "
            "payment gateway load balancer that was not flagged by automated renewal systems.\n\n"
            "Business Impact:\n"
            "- 847 merchant accounts unable to process transactions\n"
            "- Estimated revenue loss of $2.3M across affected merchants\n"
            "- 23,000 end customers experienced failed checkout flows\n"
            "- SLA breach triggered for 12 enterprise tier customers\n\n"
            "Technical Investigation:\n"
            "[14:32 UTC] Automated monitoring alerts fire for HTTP 503 on /api/v2/payments/process\n"
            "[14:38 UTC] On-call engineer pages payment infrastructure team\n"
            "[14:55 UTC] SSL certificate expiry confirmed on load balancer lb-payments-prod-01\n"
            "[15:10 UTC] Certificate renewal attempted, blocked by IAM permission misconfiguration\n"
            "[15:45 UTC] IAM permissions corrected, new certificate issued and deployed\n"
            "[18:55 UTC] Full service restoration confirmed across all merchant accounts"
        )
    }
]

inputs_rca = tokenizer.apply_chat_template(
    test_input_rca,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

print("=" * 60)
print("TASK A OUTPUT (Generate RCA):")
print("=" * 60)

outputs_rca = model.generate(
    input_ids=inputs_rca,
    max_new_tokens=1200,
    temperature=0.1,
    do_sample=True,
)

rca_output_text = tokenizer.decode(
    outputs_rca[0][inputs_rca.shape[1]:],
    skip_special_tokens=True
)
print(rca_output_text)

# ── REAL TEST: Task B ─────────────────────────────────────────
test_input_capa = [
    {
        "role": "user",
        "content": (
            "TASK: GENERATE CAPA\n\n"
            "Problem Description:\n"
            "On June 15, 2026, the enterprise payment processing gateway experienced "
            "a complete service outage lasting 4 hours and 23 minutes beginning at 14:32 UTC. "
            "The Stripe payment integration API began returning HTTP 503 errors across all "
            "merchant accounts. Transaction processing halted entirely for 847 active merchant "
            "accounts. The root trigger was traced to an expired SSL certificate on the "
            "payment gateway load balancer that was not flagged by automated renewal systems.\n\n"
            "Business Impact:\n"
            "- 847 merchant accounts unable to process transactions\n"
            "- Estimated revenue loss of $2.3M across affected merchants\n"
            "- 23,000 end customers experienced failed checkout flows\n"
            "- SLA breach triggered for 12 enterprise tier customers\n\n"
            "Technical Investigation:\n"
            "[14:32 UTC] Automated monitoring alerts fire for HTTP 503 on /api/v2/payments/process\n"
            "[14:38 UTC] On-call engineer pages payment infrastructure team\n"
            "[14:55 UTC] SSL certificate expiry confirmed on load balancer lb-payments-prod-01\n"
            "[15:10 UTC] Certificate renewal attempted, blocked by IAM permission misconfiguration\n"
            "[15:45 UTC] IAM permissions corrected, new certificate issued and deployed\n"
            "[18:55 UTC] Full service restoration confirmed across all merchant accounts\n\n"
            f"Approved Root Cause:\n{rca_output_text}"
        )
    }
]

inputs_capa = tokenizer.apply_chat_template(
    test_input_capa,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

print("\n" + "=" * 60)
print("TASK B OUTPUT (Generate CAPA):")
print("=" * 60)

outputs_capa = model.generate(
    input_ids=inputs_capa,
    max_new_tokens=1000,
    temperature=0.1,
    do_sample=True,
)

print(tokenizer.decode(
    outputs_capa[0][inputs_capa.shape[1]:],
    skip_special_tokens=True
))

Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TASK A OUTPUT (Generate RCA):


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1000) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "issue_summary": "On June 15, 2026, the enterprise payment processing gateway experienced a complete service outage lasting 4 hours and 23 minutes beginning at 14:32 UTC. The Stripe payment integration API began returning HTTP 503 errors across all merchant accounts. Transaction processing halted entirely for 847 active merchant accounts. The root trigger was traced to an expired SSL certificate on the payment gateway load balancer that was not flagged by automated renewal systems.",
  "five_why_analysis": [
    {
      "question": "1. Why did the high-level application failure block the business workflow?",
      "answer": "The payment gateway returned HTTP 503 errors, preventing merchant accounts from completing transactions."
    },
    {
      "question": "2. Why did the application layer or infrastructure nodes fail to process transactions or handle authentication?",
      "answer": "An expired SSL certificate on the load balancer prevented the gateway from establishing secure